In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd "/content/drive/MyDrive/Colab Notebooks/FinalProject/external/TPGSR"
!chmod +x scripts/setup_local_assets.sh
!bash scripts/setup_local_assets.sh --copy

/content/drive/MyDrive/Colab Notebooks/FinalProject/external/TPGSR
Prepared local TPGSR assets using mode: copy

Recognizer weights:
  /content/drive/MyDrive/Colab Notebooks/FinalProject/external/TPGSR/assets/recognizers/aster_demo.pth
  /content/drive/MyDrive/Colab Notebooks/FinalProject/external/TPGSR/assets/recognizers/moran_demo.pth
  /content/drive/MyDrive/Colab Notebooks/FinalProject/external/TPGSR/assets/recognizers/crnn.pth

TextZoom subsets:
  /content/drive/MyDrive/Colab Notebooks/FinalProject/external/TPGSR/data/TextZoom/train1
  /content/drive/MyDrive/Colab Notebooks/FinalProject/external/TPGSR/data/TextZoom/train2
  /content/drive/MyDrive/Colab Notebooks/FinalProject/external/TPGSR/data/TextZoom/test/easy
  /content/drive/MyDrive/Colab Notebooks/FinalProject/external/TPGSR/data/TextZoom/test/medium
  /content/drive/MyDrive/Colab Notebooks/FinalProject/external/TPGSR/data/TextZoom/test/hard


In [3]:
!ls -lh assets/recognizers
!du -sh data/TextZoom/train1 data/TextZoom/train2 data/TextZoom/test/easy data/TextZoom/test/medium data/TextZoom/test/hard

total 191M
-rw-------+ 1 root root 81M Apr  9 20:42 aster_demo.pth
-rw-------+ 1 root root 32M Apr  9 20:43 crnn.pth
-rw-------+ 1 root root 78M Apr  9 20:43 moran_demo.pth
85M	data/TextZoom/train1
20M	data/TextZoom/train2
13M	data/TextZoom/test/easy
5.6M	data/TextZoom/test/medium
12M	data/TextZoom/test/hard


In [4]:
!pip install -q thop ptflops tensorboardX easydict editdistance lmdb opencv-python-headless scipy pyyaml

In [5]:
from pathlib import Path
import re

def replace_once_or_skip(path_str, transform):
    path = Path(path_str)
    original = path.read_text(encoding="utf-8")
    updated = transform(original)
    if updated != original:
        path.write_text(updated, encoding="utf-8")
        print(f"patched: {path_str}")
    else:
        print(f"already ok: {path_str}")

def patch_dataset_pyfasttext(text):
    pattern = r"from IPython import embed\s+.*?random\.seed\(0\)"
    replacement = """from IPython import embed
try:
    from pyfasttext import FastText
except ImportError:
    FastText = None
random.seed(0)"""
    updated, count = re.subn(pattern, replacement, text, count=1, flags=re.S)
    if count == 1:
        return updated
    if replacement in text:
        return text
    raise RuntimeError("Could not repair pyfasttext import block in dataset/dataset.py")

def patch_utils_deblur(text):
    old = "    h[h < scipy.finfo(float).eps * h.max()] = 0\n"
    new = "    h[h < np.finfo(float).eps * h.max()] = 0\n"
    if old in text:
        return text.replace(old, new, 1)
    if new in text:
        return text
    raise RuntimeError("Could not find scipy.finfo line in utils/utils_deblur.py")

def patch_base_save_checkpoint(text):
    old = """        for i in range(len(netG_list)):
            netG = netG_list[i]
            save_dict = {
                'state_dict_G': netG.module.state_dict(),
                'info': {'arch': self.args.arch, 'iters': iters, 'epochs': epoch, 'batch_size': self.batch_size,
                         'voc_type': self.voc_type, 'up_scale_factor': self.scale_factor},
                'best_history_res': best_acc_dict,
                'best_model_info': best_model_info,
                'param_num': sum([param.nelement() for param in netG.module.parameters()]),
                'converge': converge_list,
            }
"""
    new = """        for i in range(len(netG_list)):
            netG = netG_list[i]
            netG_to_save = netG.module if hasattr(netG, 'module') else netG
            save_dict = {
                'state_dict_G': netG_to_save.state_dict(),
                'info': {'arch': self.args.arch, 'iters': iters, 'epochs': epoch, 'batch_size': self.batch_size,
                         'voc_type': self.voc_type, 'up_scale_factor': self.scale_factor},
                'best_history_res': best_acc_dict,
                'best_model_info': best_model_info,
                'param_num': sum([param.nelement() for param in netG_to_save.parameters()]),
                'converge': converge_list,
            }
"""
    if old in text:
        return text.replace(old, new, 1)
    if "netG_to_save = netG.module if hasattr(netG, 'module') else netG" in text:
        return text
    raise RuntimeError("Could not find save_checkpoint block in interfaces/base.py")

replace_once_or_skip("dataset/dataset.py", patch_dataset_pyfasttext)
replace_once_or_skip("utils/utils_deblur.py", patch_utils_deblur)
replace_once_or_skip("interfaces/base.py", patch_base_save_checkpoint)

print("Compatibility patch complete.")

already ok: dataset/dataset.py
already ok: utils/utils_deblur.py
already ok: interfaces/base.py
Compatibility patch complete.


In [6]:
from pathlib import Path

def show_block(path_str, start_marker, end_marker=None, extra_lines=0):
    text = Path(path_str).read_text(encoding="utf-8")
    start = text.index(start_marker)
    if end_marker is None:
        end = start + len(start_marker)
    else:
        end = text.index(end_marker, start) + len(end_marker)
    snippet = text[start:end]
    if extra_lines:
        lines = text[start:].splitlines()
        snippet = "\n".join(lines[:extra_lines])
    print(f"\n--- {path_str} ---")
    print(snippet)

show_block(
    "dataset/dataset.py",
    "from IPython import embed",
    "random.seed(0)"
)

show_block(
    "utils/utils_deblur.py",
    "def fspecial_gaussian",
    extra_lines=10
)

show_block(
    "interfaces/base.py",
    "def save_checkpoint",
    extra_lines=20
)

dataset_text = Path("dataset/dataset.py").read_text(encoding="utf-8")
deblur_text = Path("utils/utils_deblur.py").read_text(encoding="utf-8")
base_text = Path("interfaces/base.py").read_text(encoding="utf-8")

assert "try:\n    from pyfasttext import FastText\nexcept ImportError:\n    FastText = None" in dataset_text
assert "np.finfo(float).eps" in deblur_text
assert "netG_to_save = netG.module if hasattr(netG, 'module') else netG" in base_text

print("\nVerification passed.")


--- dataset/dataset.py ---
from IPython import embed
try:
    from pyfasttext import FastText
except ImportError:
    FastText = None
random.seed(0)

--- utils/utils_deblur.py ---
def fspecial_gaussian(hsize, sigma):
    hsize = [hsize, hsize]
    siz = [(hsize[0]-1.0)/2.0, (hsize[1]-1.0)/2.0]
    std = sigma
    [x, y] = np.meshgrid(np.arange(-siz[1], siz[1]+1), np.arange(-siz[0], siz[0]+1))
    arg = -(x*x + y*y)/(2*std*std)
    h = np.exp(arg)
    h[h < np.finfo(float).eps * h.max()] = 0
    sumh = h.sum()
    if sumh != 0:

--- interfaces/base.py ---
def save_checkpoint(self, netG_list, epoch, iters, best_acc_dict, best_model_info, is_best, converge_list, recognizer=None):
        ckpt_path = os.path.join('ckpt', self.vis_dir)
        if not os.path.exists(ckpt_path):
            os.mkdir(ckpt_path)

        print("Into saving checkpoints...")

        for i in range(len(netG_list)):
            netG = netG_list[i]
            netG_to_save = netG.module if hasattr(netG, 'module') 

In [7]:
from pathlib import Path
import yaml

path = Path("config/super_resolution.local.yaml")
data = yaml.safe_load(path.read_text(encoding="utf-8"))

data["TRAIN"]["epochs"] = 200
data["TRAIN"]["valInterval"] = 361
data["TRAIN"]["saveInterval"] = 361
data["TRAIN"]["displayInterval"] = 361

path.write_text(yaml.safe_dump(data, sort_keys=False), encoding="utf-8")
print("Updated config:")
print("epochs =", data["TRAIN"]["epochs"])
print("valInterval =", data["TRAIN"]["valInterval"])
print("saveInterval =", data["TRAIN"]["saveInterval"])
print("displayInterval =", data["TRAIN"]["displayInterval"])

Updated config:
epochs = 240
valInterval = 361
saveInterval = 361
displayInterval = 361


In [8]:
!bash train_TPGSR-TSRN.local.sh

We have  14573 samples...
We have  2794 samples...
We have  1619 samples...
We have  1411 samples...
We have  1343 samples...
TSRN_TL(
  3.55 M, 100.000% Params, 971.1 MMac, 99.984% MACs, 
  (block1): Sequential(
    20.8 k, 0.587% Params, 21.36 MMac, 2.200% MACs, 
    (0): Conv2d(20.8 k, 0.587% Params, 21.3 MMac, 2.193% MACs, 4, 64, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4))
    (1): PReLU(1, 0.000% Params, 65.54 KMac, 0.007% MACs, num_parameters=1)
  )
  (block2): RecurrentResidualBlockTL(
    122.11 k, 3.444% Params, 125.96 MMac, 12.969% MACs, 
    (conv1): Conv2d(36.93 k, 1.041% Params, 37.81 MMac, 3.893% MACs, 64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn1): BatchNorm2d(128, 0.004% Params, 131.07 KMac, 0.013% MACs, 64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (gru1): GruBlock(
      25.02 k, 0.706% Params, 26.08 MMac, 2.686% MACs, 
      (conv1): Conv2d(6.21 k, 0.175% Params, 6.36 MMac, 0.655% MACs, 96, 64, kernel_size=(1, 1)